In [0]:
# Create Bronze schema

spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.bronze
""")

print("Bronze schema created successfully!")

Bronze schema created successfully!


In [0]:
spark.sql("""
SHOW SCHEMAS IN workspace
""").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|information_schema|
|students_intro    |
+------------------+



In [0]:
# Source and destination configuration

SOURCE_PATH = "/Volumes/workspace/default/olist_raw/"

TARGET_SCHEMA = "workspace.bronze"

# Mapping of source CSV files to Bronze table names

source_tables = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "payments",
    "olist_order_reviews_dataset.csv": "reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation"
}

print("Total source datasets:", len(source_tables))

Total source datasets: 9


In [0]:

from pyspark.sql import functions as F

def ingest_to_bronze(file_name, table_name):

    # Read source CSV
    df = spark.read.csv(
        SOURCE_PATH + file_name,
        header=True,
        inferSchema=True
    )

    # Add ingestion metadata
    df = (
        df.withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_file",
            F.col("_metadata.file_path")
        )
    )

    # Define target table
    target_table = f"{TARGET_SCHEMA}.{table_name}"

    # Write data as Delta table
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table)

    print(f"Successfully ingested: {target_table}")

    return df.count()


In [0]:
ingestion_results = {}

for file_name, table_name in source_tables.items():

    count = ingest_to_bronze(
        file_name,
        table_name
    )

    ingestion_results[table_name] = count

print("\nBRONZE INGESTION SUMMARY")

for table, count in ingestion_results.items():
    print(f"{table}: {count} records")

Successfully ingested: workspace.bronze.customers
Successfully ingested: workspace.bronze.geolocation
Successfully ingested: workspace.bronze.order_items
Successfully ingested: workspace.bronze.payments
Successfully ingested: workspace.bronze.reviews
Successfully ingested: workspace.bronze.orders
Successfully ingested: workspace.bronze.products
Successfully ingested: workspace.bronze.sellers
Successfully ingested: workspace.bronze.category_translation

BRONZE INGESTION SUMMARY
customers: 99441 records
geolocation: 1000163 records
order_items: 112650 records
payments: 103886 records
reviews: 104162 records
orders: 99441 records
products: 32951 records
sellers: 3095 records
category_translation: 71 records


In [0]:
# List all tables in Bronze schema

spark.sql("""
SHOW TABLES IN workspace.bronze
""").show(truncate=False)

+--------+--------------------+-----------+
|database|tableName           |isTemporary|
+--------+--------------------+-----------+
|bronze  |category_translation|false      |
|bronze  |customers           |false      |
|bronze  |geolocation         |false      |
|bronze  |order_items         |false      |
|bronze  |orders              |false      |
|bronze  |payments            |false      |
|bronze  |products            |false      |
|bronze  |reviews             |false      |
|bronze  |sellers             |false      |
+--------+--------------------+-----------+



In [0]:
for table_name in source_tables.values():

    full_table_name = f"workspace.bronze.{table_name}"

    count = spark.table(full_table_name).count()

    print(f"{table_name}: {count} records")

customers: 99441 records
geolocation: 1000163 records
order_items: 112650 records
payments: 103886 records
reviews: 104162 records
orders: 99441 records
products: 32951 records
sellers: 3095 records
category_translation: 71 records


In [0]:
orders_bronze = spark.table(
    "workspace.bronze.orders"
)

# Display first 10 records

display(orders_bronze.limit(10))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z,2026-11-14T17:50:23.202Z,dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv


In [0]:
spark.sql("""
DESCRIBE DETAIL workspace.bronze.orders
""").show(truncate=False)

+------+------------------------------------+-----------------------+-----------+--------+----------------------+-------------------+----------------+-----------------+--------+-----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+------------------------------------------------------------------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                   |description

In [0]:
spark.sql("""
SELECT
    order_id,
    order_status,
    ingestion_timestamp,
    source_file
FROM workspace.bronze.orders
LIMIT 5
""").show(truncate=False)

+--------------------------------+------------+--------------------------+------------------------------------------------------------------+
|order_id                        |order_status|ingestion_timestamp       |source_file                                                       |
+--------------------------------+------------+--------------------------+------------------------------------------------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|delivered   |2026-11-14 17:50:23.202747|dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv|
|53cdb2fc8bc7dce0b6741e2150273451|delivered   |2026-11-14 17:50:23.202747|dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv|
|47770eb9100c2d0c44946d9cf07ec65d|delivered   |2026-11-14 17:50:23.202747|dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv|
|949d5b44dbf5de918fe9c16f97b45f8a|delivered   |2026-11-14 17:50:23.202747|dbfs:/Volumes/workspace/default/olist_raw/olist_orders_dataset.csv|
|ad21c

In [0]:
# Create Bronze ingestion audit schema

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.ingestion_audit (
    table_name STRING,
    source_file STRING,
    record_count BIGINT,
    ingestion_timestamp TIMESTAMP,
    ingestion_status STRING
)
USING DELTA
""")

print("Ingestion audit table created!")

Ingestion audit table created!


In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

# Prepare audit records

audit_records = []

for file_name, table_name in source_tables.items():

    audit_records.append(
        Row(
            table_name=table_name,
            source_file=file_name,
            record_count=ingestion_results[table_name],
            ingestion_status="SUCCESS"
        )
    )

# Convert records into a Spark DataFrame

audit_df = spark.createDataFrame(audit_records)

# Add ingestion timestamp

audit_df = audit_df.withColumn(
    "ingestion_timestamp",
    F.current_timestamp()
)

# Write audit records

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.bronze.ingestion_audit")

print("Audit records saved successfully!")

Audit records saved successfully!


In [0]:
display(
    spark.table("workspace.bronze.ingestion_audit")
)

table_name,source_file,record_count,ingestion_timestamp,ingestion_status
customers,olist_customers_dataset.csv,99441,2026-11-14T17:55:41.969Z,SUCCESS
geolocation,olist_geolocation_dataset.csv,1000163,2026-11-14T17:55:41.969Z,SUCCESS
order_items,olist_order_items_dataset.csv,112650,2026-11-14T17:55:41.969Z,SUCCESS
payments,olist_order_payments_dataset.csv,103886,2026-11-14T17:55:41.969Z,SUCCESS
reviews,olist_order_reviews_dataset.csv,104162,2026-11-14T17:55:41.969Z,SUCCESS
orders,olist_orders_dataset.csv,99441,2026-11-14T17:55:41.969Z,SUCCESS
products,olist_products_dataset.csv,32951,2026-11-14T17:55:41.969Z,SUCCESS
sellers,olist_sellers_dataset.csv,3095,2026-11-14T17:55:41.969Z,SUCCESS
category_translation,product_category_name_translation.csv,71,2026-11-14T17:55:41.969Z,SUCCESS
